# Attention-Pattern Change Under Intervention

Measures how much each attention head's attention pattern — not its output — shifts toward the intervened content when the source prompt is spliced into the base prompt. Compares attention weight at the intervention position and at a nearby reference position, before and after the splice.

In [1]:
%load_ext autoreload
%autoreload 2

### Setup

Imports `_config`/`_prompt`/`_mapping` and checks GPU availability.

In [ ]:
import sys
sys.path.append("src")

import torch
import gc
from tqdm import tqdm

import _config
import _prompt
import _mapping

In [3]:
_util.print_GPU_availbility()

CUDA is available: True
Available devices:
  GPU 0: NVIDIA RTX A5500
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B

## Experiment Config And Model Loading

In [4]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS_stepwise", # GPT-OSS or R1
    prompt_type="h_pre_penultimate_sum", # empty or pre_result or pre_sum
)
intervention_config = _config.InterventionConfig(
    intervention_loc="explicit_ids",
    intervention_ids=[20],
    tok_pos_fn=_mapping.intervene_id_to_tok_pos_stepwise_3_digit_h,
)
run_config = _config.RunConfig(
    experiment_root="experiments/attention_heads",
    result_dir="pattern_diff",
    output_filename=f"{prompt_config.stem}.csv",
)
batch_size = 4

model, tokenizer = _config.load_model(prompt_config.model_type)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Load Prompts

Loads the configured base/source prompt pairs to run the sweep over.

In [5]:
prompts = _config.load_prompts(prompt_config)
print(f"loaded {len(prompts)} prompts")

loaded 256 divided prompts


## Resolve Intervention Ids And Token Positions

In [6]:
intervention_ids = list(intervention_config.intervention_ids)
tok_pos_list = _config.build_tok_pos_list(intervention_config.tok_pos_fn, intervention_ids)
print(intervention_ids)
print(tok_pos_list)

[20]
[235]


## Run Pattern-Diff Sweep

For each prompt batch, runs the model on the clean and the intervened prompt, and records the change in attention weight at the intervention position and at a nearby reference position, for every layer and head.

In [7]:
# Get header of prompts dataset
header = list(prompts.columns) + ['layer', 'head_num', 'faithful_diff', 'unfaithful_diff']
filepath = _config.build_run_output_filepath(prompt_config, run_config, header)

for i in tqdm(range(0, len(prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = prompts.iloc[i:i+batch_size]
    
    # Tokenize all prompts in the batch
    clean_tokens = tokenizer(batch_rows['base_prompt'].tolist(), add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    # Generate for the entire batch
    clean_output = model(
        clean_tokens.input_ids,
        attention_mask=clean_tokens.attention_mask,
        pad_token_id=tokenizer.pad_token_id,
        output_attentions=True,
    )
    # Pick out relevant attention patterns
    clean_faithful_attentions = torch.stack([layer_attention[:,:,-1,tok_pos_list[0]] for layer_attention in clean_output.attentions])
    clean_unfaithful_attentions = torch.stack([layer_attention[:,:,-1,tok_pos_list[0]-2] for layer_attention in clean_output.attentions])
    
    # Prepare batch of intervention prompts
    intervened_prompts = _config.build_intervened_prompts(batch_rows, intervention_ids)
    # Tokenize all prompts in the batch
    intervened_tokens = tokenizer(intervened_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    # Generate for the entire batch
    intervened_output = model(
        intervened_tokens.input_ids,
        attention_mask=intervened_tokens.attention_mask,
        pad_token_id=tokenizer.pad_token_id,
        output_attentions=True,
    )
    # Pick out relevant attention patterns
    intervened_faithful_attentions = torch.stack([layer_attention[:,:,-1,tok_pos_list[0]] for layer_attention in intervened_output.attentions])
    intervened_unfaithful_attentions = torch.stack([layer_attention[:,:,-1,tok_pos_list[0]-2] for layer_attention in intervened_output.attentions])
    
    attention_faithful_diffs = intervened_faithful_attentions - clean_faithful_attentions
    attention_unfaithful_diffs = intervened_unfaithful_attentions - clean_unfaithful_attentions
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        for layer in range(model.config.num_hidden_layers):
            for head in range(model.config.num_attention_heads):
                _config.write_to_csv(filepath, row.to_list() + [layer, head, attention_faithful_diffs[layer, j, head].item(), attention_unfaithful_diffs[layer, j, head].item()])

    del clean_tokens, clean_output, intervened_tokens, intervened_output, clean_faithful_attentions, clean_unfaithful_attentions, intervened_faithful_attentions, intervened_unfaithful_attentions, attention_faithful_diffs, attention_unfaithful_diffs
    torch.cuda.empty_cache()
    gc.collect()


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [16:16<00:00, 15.27s/it]
